In [0]:
spark.read.table("workspace.silver.silver_people").printSchema()

In [0]:
from pyspark.sql.functions import col, count, floor, datediff, current_date

# 1. Load the Silver data
df_silver = spark.read.table("workspace.silver.silver_people")

# 2. Create the Gold Aggregation: Job Title Distribution
df_gold_jobs = df_silver.groupBy("job_title") \
    .agg(count("user_id").alias("total_employees")) \
    .orderBy(col("total_employees").desc())

# 3. Create a Gold Demographic view (Age Calculation)
df_gold_demographics = df_silver.withColumn("age", 
    floor(datediff(current_date(), col("date_of_birth")) / 365.25)) \
    .groupBy("sex") \
    .agg(count("user_id").alias("count"), 
         floor(col("age").cast("double")).alias("avg_age"))

# Show the results
print("Gold Table: Job Distribution")
display(df_gold_jobs)

print("Gold Table: Demographic Summary")
display(df_gold_demographics)

In [0]:
from pyspark.sql.functions import col, count, floor, datediff, current_date, avg, round

# 1. Create the 'age' column and store it in df_with_age
df_with_age = df_silver.withColumn("age", 
    floor(datediff(current_date(), col("date_of_birth")) / 365.25))

# 2. Now use df_with_age to create the summary
df_gold_demographics = df_with_age.groupBy("sex") \
    .agg(
        count("user_id").alias("total_count"), 
        round(avg("age"), 1).alias("average_age")
    )

# 3. View the result
display(df_gold_demographics)

In [0]:
# 1. Save the Job Distribution Table
df_gold_jobs.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.fact_job_distribution")

# 2. Save the Demographic Table
df_gold_demographics.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.fact_demographics")

print("Pipeline Complete! Both Gold tables are now live in workspace.gold.")

In [0]:
display(spark.sql("SHOW TABLES IN workspace.gold"))

In [0]:
%sql
ALTER TABLE gold.fact_demographics RENAME TO gold.gold_demographics;
ALTER TABLE gold.fact_job_distribution RENAME TO gold.gold_job_distribution;